# 5-2. Forecasting Continuous Earnings with XGBoost

Our previous session 5-1 asked a binary question: will next year's earnings go *up or down*? This session asks the harder question: **by how much** — a continuous forecast of next year's earnings per share (EPS). We follow:

> Chattopadhyay, A., B. Fang, and P. Mohanram (2025). Machine learning for earnings forecasting - US and international evidence. Working paper. https://dx.doi.org/10.2139/ssrn.5941658

Chattopadhyay et al. (2025) benchmark XGBoost against a naive random-walk (RW) model, theory-driven linear models (HVZ, EP, RI), and penalized linear regressions (Lasso, Ridge) across US and international samples. Their central, slightly humbling finding: **a naive random walk — "next year's EPS = this year's EPS" — is a genuinely hard benchmark to beat**, especially outside the US. XGBoost wins on average, but not everywhere, and not by as much as you might expect from a "black box" model.

We replicate this benchmarking exercise at class scale, reusing 5-1's exact data (`data/comp_sample.csv`), sample filters, and feature-engineering pipeline (current value, lagged value, % change for every auto-detected financial-statement column) — the only thing that changes is the *target*: instead of a 0/1 direction label, we now predict next year's EPS directly, and instead of ROC-AUC we use the paper's own accuracy metrics (MAFE, RMSE, MDAFE, all scaled by price).

**Prerequisite:** this notebook assumes you've been through 5-1. Section 1 recreates that notebook's data and feature pipeline in condensed form — see 5-1 if any step is unclear.

## Learning objectives

By the end of this class, you will be able to:

- Frame a forecasting problem as a regression task (continuous target) rather than a classification task.
- Recognize why a naive random-walk forecast is a nontrivial benchmark for earnings prediction, and implement it in one line.
- Explain why regression targets built from raw accounting data often need outlier treatment (winsorizing) before a linear model can be trusted with them, and why tree ensembles are more forgiving.
- Fit `xgboost.XGBRegressor` with a **Huber loss** objective to be robust to outlier firm-years, and tune it with early stopping.
- Evaluate regression forecasts with mean absolute forecast error (MAFE), root mean squared error (RMSE), and median absolute forecast error (MDAFE), each scaled by price — the metrics used in the earnings-forecasting literature.
- Read an XGBoost regression feature-importance plot and relate it to the paper's finding that current earnings, cash flow, and book equity dominate.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV

import xgboost as xgb

pd.set_option('display.max_columns', 50)

## 1. Recap: load, filter, and engineer features (same as 5-1)

Same filters, same auto-detected current/lag/%-change feature pipeline as 5-1 — condensed into two cells. Nothing here is new.

In [ ]:
df = pd.read_csv('data/comp_sample.csv')

df = df[
    (df['indfmt'] == 'INDL') &
    (df['curcd'] == 'USD') &
    (df['costat'] == 'A')
].copy()

df = df.dropna(subset=['gvkey', 'fyear']).copy()
df['fyear'] = df['fyear'].astype(int)
df = df.sort_values(['gvkey', 'fyear']).reset_index(drop=True)

ID_COLS = ['gvkey', 'datadate', 'fyear']
LABEL_COLS = ['eps', 'eps_lead']  # this notebook's target-construction columns
NON_FS_COLS = ['au']
NO_SCALE_COLS = ['at', 'csho', 'prcc_f']

exclude_cols = set(ID_COLS + LABEL_COLS + NON_FS_COLS)
numeric_cols = df.select_dtypes(include='number').columns
predictor_base_cols = [c for c in numeric_cols if c not in exclude_cols]

print(df.shape)
print(f'{len(predictor_base_cols)} raw financial-statement columns auto-detected as predictors')

In [ ]:
def build_features(data: pd.DataFrame, base_cols: list[str]) -> pd.DataFrame:
    """Current value, lagged value, and percentage change for each base column."""
    at_cur = data['at']
    at_lag = data.groupby('gvkey')['at'].shift(1)

    feature_cols = {}
    for col in base_cols:
        cur = data[col]
        lag = data.groupby('gvkey')[col].shift(1)

        if col in NO_SCALE_COLS:
            cur_scaled, lag_scaled = cur, lag
        else:
            cur_scaled = (cur / at_cur).replace([np.inf, -np.inf], np.nan)
            lag_scaled = (lag / at_lag).replace([np.inf, -np.inf], np.nan)

        pct_change = ((cur - lag) / lag.abs()).replace([np.inf, -np.inf], np.nan)

        feature_cols[f'{col}_cur'] = cur_scaled
        feature_cols[f'{col}_lag'] = lag_scaled
        feature_cols[f'{col}_pctchg'] = pct_change

    return pd.DataFrame(feature_cols, index=data.index)


X_all = build_features(df, predictor_base_cols)
feature_cols = X_all.columns.tolist()
print(X_all.shape)

## 2. Build a continuous earnings target

Instead of 5-1's drift-adjusted direction label, we now predict next year's EPS directly, following Chattopadhyay et al. (2025):

$$EPS_t = \frac{NI_t}{CSHO_t}, \qquad \text{target}_t = EPS_{t+1}$$

We also keep two columns we don't use as *predictors* but do need for the rest of the notebook: `prcc_f` (fiscal-year-end share price), which every forecast error below is scaled by, and current-year `eps`, which *is* the random-walk forecast of next year's EPS (Section 4).

Two data-cleaning steps mirror the paper directly:

- **Require a $1 minimum share price.** Chattopadhyay et al. (2025, Section 3.5.1) require "the stock price at the end of June to exceed 1 USD" in their US sample. We do the same — without it, a handful of sub-cent "penny stock" observations make the price-scaled error metrics in Section 7 explode, since dividing by a price near zero turns even a small dollar miss into an enormous *percentage-of-price* miss.
- **Winsorize the target.** Raw EPS has a few extreme outliers (a firm with an unusual one-off gain or loss can have EPS in the thousands). We cap `eps` and `eps_lead` at the 1st/99th percentile of the **training period only** — never the validation or test period, for the same look-ahead-bias reason 5-1 fits the imputer and scaler on training data only.

In [ ]:
df['eps'] = (df['ni'] / df['csho']).replace([np.inf, -np.inf], np.nan)
df['eps_lead'] = df.groupby('gvkey')['eps'].shift(-1)

model_df = pd.concat(
    [df[['gvkey', 'fyear', 'eps', 'eps_lead', 'prcc_f']], X_all], axis=1
)
model_df = model_df.dropna(subset=['eps', 'eps_lead', 'prcc_f']).reset_index(drop=True)
model_df = model_df[model_df['prcc_f'] >= 1].reset_index(drop=True)

print(model_df.shape)
model_df[['gvkey', 'fyear', 'eps', 'eps_lead', 'prcc_f']].head(10)

## 3. Split chronologically (same cutoffs as 5-1), then winsorize using training bounds only

Same reasoning as 5-1: a random split would leak future information into training. We reuse the identical `TRAIN_END` / `VAL_END` cutoffs, then compute the 1st/99th-percentile winsorizing bounds from the training rows only and apply them to every split — including to the random-walk forecast itself, so all models are evaluated against the same (winsorized) yardstick.

In [ ]:
TRAIN_END = 2019
VAL_END = 2021

train_mask = model_df['fyear'] <= TRAIN_END
lo, hi = model_df.loc[train_mask, 'eps_lead'].quantile([0.01, 0.99])
print(f'Winsorizing bounds from the training period: [{lo:.2f}, {hi:.2f}]')

model_df['eps_lead'] = model_df['eps_lead'].clip(lo, hi)
model_df['eps'] = model_df['eps'].clip(lo, hi)

train = model_df[model_df['fyear'] <= TRAIN_END]
val = model_df[(model_df['fyear'] > TRAIN_END) & (model_df['fyear'] <= VAL_END)]
test = model_df[model_df['fyear'] > VAL_END]

X_train, y_train = train[feature_cols], train['eps_lead']
X_val, y_val = val[feature_cols], val['eps_lead']
X_test, y_test = test[feature_cols], test['eps_lead']

# Needed for benchmarks and price-scaled evaluation (Sections 4-7), not as model features
rw_test = test['eps']            # random-walk forecast: next year's EPS = this year's EPS
price_test = test['prcc_f']

print(f'train: {X_train.shape}, val: {X_val.shape}, test: {X_test.shape}')

## 4. Benchmark 1: the random walk

The simplest possible forecast: assume next year's EPS equals this year's EPS. Chattopadhyay et al. (2025) show this is *not* a strawman — outside the US it beats every linear model they try, and even in the US it is competitive with theory-driven cross-sectional models. It costs nothing to compute, which is exactly the point: any model we build should have to earn its complexity by beating this.

In [ ]:
rw_pred = rw_test.to_numpy()  # no fitting required
print("Random-walk forecast is just last year's EPS -- no model to fit.")

## 5. Benchmark 2: regularized linear regression

Chattopadhyay et al. (2025, Section 3.1.2) are explicit about why they don't use plain OLS here: "With many potential predictors, OLS is prone to overfitting: it may fit the training data well but perform poorly out of sample." Our 27 engineered features include several near-duplicates (a column's current value, its lag, and its % change are all correlated) — exactly the condition that makes unregularized OLS unstable. So, like the paper, we use a **penalized** linear model: `RidgeCV`, which shrinks coefficients toward zero and automatically picks the shrinkage strength via built-in cross-validation.

As with 5-1's logistic benchmark, we median-impute and standardize first, since linear models can't take NaNs and benefit from comparable feature scales.

In [ ]:
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train)
X_val_imp = imputer.transform(X_val)
X_test_imp = imputer.transform(X_test)

scaler = StandardScaler()
X_train_imp = scaler.fit_transform(X_train_imp)
X_val_imp = scaler.transform(X_val_imp)
X_test_imp = scaler.transform(X_test_imp)

ridge = RidgeCV(alphas=np.logspace(-2, 4, 25))
ridge.fit(X_train_imp, y_train)

ridge_pred = ridge.predict(X_test_imp)
print(f'Ridge benchmark fitted (selected alpha = {ridge.alpha_:.2f}).')

## 6. XGBoost

### 6.1 A baseline model

Chattopadhyay et al. (2025) use a **Huber loss** (`reg:pseudohubererror`) instead of the default squared-error loss, specifically to make XGBoost more robust to the occasional extreme EPS value — the tree-ensemble analogue of the winsorizing we had to do by hand for Ridge in Section 3. As in 5-1, XGBoost takes `X_train` with its NaNs untouched — no imputation needed.

In [ ]:
xgb_baseline = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='reg:pseudohubererror',
    eval_metric='mae',
    random_state=42,
)
xgb_baseline.fit(X_train, y_train)

baseline_pred = xgb_baseline.predict(X_test)
print('XGBoost (untuned baseline) fitted.')

### 6.2 Tuning with early stopping

Same idea as 5-1: grow up to 2,000 trees, but stop once validation MAE stops improving for 50 rounds in a row.

In [ ]:
xgb_tuned = xgb.XGBRegressor(
    n_estimators=2000,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='reg:pseudohubererror',
    eval_metric='mae',
    early_stopping_rounds=50,
    random_state=42,
)
xgb_tuned.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False,
)
print(f'Stopped after {xgb_tuned.best_iteration + 1} trees '
      f'(validation MAE = {xgb_tuned.best_score:.3f})')

tuned_pred = xgb_tuned.predict(X_test)

## 7. Evaluate: MAFE, RMSE, and MDAFE, scaled by price

Following Chattopadhyay et al. (2025, Section 3.3), we evaluate every model with three complementary metrics, each computed on the **absolute forecast error scaled by price**:

- **MAFE** (mean absolute forecast error): average error magnitude — sensitive to every observation equally.
- **RMSE** (root mean squared error): penalizes large misses more heavily than MAFE.
- **MDAFE** (median absolute forecast error): robust to outliers, describing the "typical" firm's forecast error.

We also winsorize the scaled *errors* themselves at the 1st and 99th percentiles before aggregating, exactly as the paper does, "to minimize the effect of outliers."

In [ ]:
def forecast_metrics(y_true, y_pred, price) -> dict[str, float]:
    scaled_error = (np.asarray(y_pred) - np.asarray(y_true)) / np.asarray(price)
    abs_error = pd.Series(np.abs(scaled_error))
    signed_error = pd.Series(scaled_error)
    abs_error_w = abs_error.clip(*abs_error.quantile([0.01, 0.99]))
    signed_error_w = signed_error.clip(*signed_error.quantile([0.01, 0.99]))
    return {
        'MAFE': abs_error_w.mean(),
        'RMSE': np.sqrt((signed_error_w ** 2).mean()),
        'MDAFE': abs_error_w.median(),
    }


results = pd.DataFrame({
    'Random walk': forecast_metrics(y_test, rw_pred, price_test),
    'Ridge regression': forecast_metrics(y_test, ridge_pred, price_test),
    'XGBoost (baseline)': forecast_metrics(y_test, baseline_pred, price_test),
    'XGBoost (early-stopped)': forecast_metrics(y_test, tuned_pred, price_test),
}).T

results.round(3)

In [ ]:
results['MAFE'].sort_values().plot(kind='barh')
plt.xlabel('Mean absolute forecast error (scaled by price)')
plt.title('Lower is better: MAFE by model')
plt.tight_layout()
plt.show()

## 8. Which predictors matter most?

Same gain-based `feature_importances_` plot as 5-1, now for a regression model. Chattopadhyay et al. (2025, Section 4.2.1) find that current earnings dominate for one-year-ahead forecasts, with cash flow from operations and book equity also consistently important — see if the same items show up here, keeping in mind our feature set is far smaller than the paper's 56 (or more) predictors.

In [ ]:
importances = pd.Series(
    xgb_tuned.feature_importances_, index=feature_cols
).sort_values(ascending=False)

top_n = 20
plt.figure(figsize=(8, 6))
importances.head(top_n).sort_values().plot(kind='barh')
plt.xlabel('XGBoost feature importance (gain)')
plt.title(f'Top {top_n} predictors of next-year EPS')
plt.tight_layout()
plt.show()

## 9. Wrap-up and discussion

**How do we compare to the paper?** Chattopadhyay et al. (2025) find that in the US, XGBoost's MAFE is about 5% lower than the next-best model at the one-year horizon, and that XGBoost is the *only* model to consistently beat the random walk both in the US and internationally — though the random walk remains a genuinely tough benchmark, especially outside the US. Look at the table in Section 7: does XGBoost beat the random walk here? Does the linear regression benchmark?

**What did we simplify, relative to the paper?**

- **Benchmarks**: a single Ridge regression instead of the paper's theory-driven HVZ/EP/RI models and both Lasso and Ridge.
- **Predictors**: a few dozen Compustat items (current, lag, % change) instead of the paper's ~56-60 predictors plus macroeconomic variables (GDP growth, unemployment, etc.).
- **Sample**: US-only, one panel (2014-2023), instead of the paper's 1969-2020 US sample and 53-country international sample — so we can't speak to the paper's central finding that XGBoost's advantage over the random walk is much smaller internationally.
- **Evaluation**: we stop at forecast-accuracy metrics. The paper goes further, validating "value relevance" by checking whether forecasts explain realized stock returns and produce sensible implied-cost-of-capital estimates (Sections 3.4 and 4.3) — a natural extension if you have CRSP return data handy.

**Discussion / exercise ideas**

1. Replace `RidgeCV` with plain `sklearn.linear_model.LinearRegression` (no regularization) and refit Section 5 — how much worse does the test-set MAFE get, and why? Then try `LassoCV` instead of `RidgeCV` — does it select a meaningfully different set of important features (check `.coef_`)?
2. The paper finds XGBoost's advantage over the random walk is largest for small firms, loss firms, and firms with volatile earnings (Section 4.1.3). Split the test set by firm size (e.g., median `at_cur`) or by whether `eps` is negative, and recompute MAFE for each model within each group — does the same pattern show up here?
3. Try predicting *scaled* earnings (`eps_lead / at_cur`, i.e., earnings scaled by assets) instead of raw EPS — the paper reports this as a robustness check (their online appendix Table OA10). Does the ranking of models change?
4. Section 8's feature-importance plot uses gain, exactly like 5-1. If you've been through 5-3o, try computing SHAP values for `xgb_tuned` here — does the *direction* of each feature's effect on predicted EPS make economic sense?